# Decoding Your DNA — Workshop Notebook

This notebook walks through building a genome analysis pipeline from scratch.

**What we'll cover:**
1. Loading and exploring the raw data
2. Looking up specific SNPs manually
3. Curated SNP analysis (drug metabolism, cardiovascular, nutrition…)
4. Full ClinVar disease risk scan
5. PharmGKB drug interaction lookup
6. Running the full pipeline and reading the reports

In [ ]:
import sys
import csv
import json
import subprocess
import time
import importlib
from pathlib import Path
from collections import defaultdict, Counter

sys.path.insert(0, str(Path("src").resolve()))

REPO_ROOT = Path(".").resolve()
DATA_DIR  = REPO_ROOT / "data"

# ─── Configure your genome file path here ──────────────────────────────────
# GENOME_PATH = DATA_DIR / Path("genome_Melinda_Chaperlo_v5_Full_20240730223601.txt")  # ← change this
# GENOME_PATH = DATA_DIR / Path("genome_Daniel_Munro_Full_20141013190727.txt")
GENOME_PATH = DATA_DIR / Path("genome_John_Doe_Full_20160201142858.txt")
# ───────────────────────────────────────────────────────────────────────────

import workshop_helpers
importlib.reload(workshop_helpers)
from workshop_helpers import *  # noqa: F401,F403

print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {DATA_DIR}")
print(f"Genome    : {GENOME_PATH}  (exists={GENOME_PATH.exists()})")

---
## 1 — Loading the Genome

The 23andMe file is a plain TSV. We build **two indexes** for O(1) lookups:
- `genome_by_rsid` — look up any named SNP (`rs762551`)
- `genome_by_position` — match ClinVar rows by `chrom:pos`

```
rsid          chromosome  position  genotype
rs4477212     1           72017     AA
rs12124819    1           766409    AG
i3002026      1           1346746   --    ← no-call: chip couldn't read this position
```

In [ ]:
t0 = time.time()
genome_by_rsid, genome_by_position, no_calls = load_genome(GENOME_PATH)
print(f"Loaded {len(genome_by_rsid):,} SNPs in {time.time()-t0:.2f}s  ({no_calls:,} no-calls skipped)")

### 1.1 — SNP distribution across chromosomes

In [ ]:
plot_chromosome_distribution(genome_by_rsid)

---
## 2 — Individual SNP Lookups

A SNP lookup is just a dictionary key. We start with **APOE** — the most clinically significant SNP for Alzheimer's disease and cardiovascular risk.

APOE status is a **haplotype**: it requires combining two SNPs into one interpretation:

| rs429358 | rs7412 | APOE allele |
|----------|--------|-------------|
| T        | T      | ε2 |
| T        | C      | ε3 |
| C        | C      | ε4 |

In [ ]:
apoe_risk = {
    ("TT", "TT"): ("ε2/ε2", "Lowest Alzheimer's risk; possible cardiovascular protection"),
    ("TT", "TC"): ("ε2/ε3", "Below-average Alzheimer's risk"),
    ("TT", "CC"): ("ε2/ε4", "Slightly elevated Alzheimer's risk"),
    ("TC", "TC"): ("ε3/ε3", "Average population risk (most common)"),
    ("TC", "CC"): ("ε3/ε4", "~3x increased Alzheimer's risk"),
    ("CC", "CC"): ("ε4/ε4", "~12x increased Alzheimer's risk"),
}

apoe1 = genome_by_rsid.get("rs429358", {}).get("genotype", "?")
apoe2 = genome_by_rsid.get("rs7412",   {}).get("genotype", "?")
result = apoe_risk.get((apoe1, apoe2)) or apoe_risk.get((apoe2, apoe1))

print(f"rs429358 (APOE): {apoe1}")
print(f"rs7412   (APOE): {apoe2}")
if result:
    allele, interpretation = result
    print(f"\nAPOE status : {allele}")
    print(f"Meaning     : {interpretation}")
else:
    print("\nCould not determine APOE status.")

### 2.1 — More famous SNPs

The same one-line lookup works for any rsID:

In [ ]:
snps_of_interest = [
    ("rs3892097", "CYP2D6 — metabolizes ~25% of all drugs"),
    ("rs4244285", "CYP2C19 — clopidogrel, PPIs, antidepressants"),
    ("rs1799853", "CYP2C9 — warfarin sensitivity"),
    ("rs1801133", "MTHFR — folate / methylation"),
    ("rs6025",    "Factor V Leiden — blood clotting"),
    ("rs762551",  "CYP1A2 — caffeine metabolism"),
]
display_snp_lookups(genome_by_rsid, snps_of_interest)

---
## 3 — Curated SNP Analysis

~200 high-impact SNPs with pre-interpreted genotypes, spanning drug metabolism, cardiovascular, methylation, fitness, sleep, and cognition.

Each entry has a **magnitude** (0–4): normal → low → moderate → high → critical.

### 3.1 — Run the analysis

For each curated SNP, we look up your genotype and check both strand orientations — 23andMe reports the forward strand, but databases vary, so `AG` and `GA` are both checked.

In [ ]:
from analyze_dna.comprehensive_snp_database import COMPREHENSIVE_SNPS

def analyze_curated_snps(genome: dict, snp_db: dict) -> list:
    findings = []
    for rsid, info in snp_db.items():
        if rsid not in genome:
            continue
        genotype     = genome[rsid]["genotype"]
        genotype_rev = genotype[::-1]          # AG → GA: check both strand orientations

        variant_info = info["variants"].get(genotype) or info["variants"].get(genotype_rev)
        if variant_info:
            findings.append({
                "rsid":        rsid,
                "gene":        info["gene"],
                "category":    info["category"],
                "genotype":    genotype,
                "status":      variant_info["status"],
                "description": variant_info["desc"],
                "magnitude":   variant_info["magnitude"],
            })
    return findings

findings = analyze_curated_snps(genome_by_rsid, COMPREHENSIVE_SNPS)
print(f"{len(findings)} / {len(COMPREHENSIVE_SNPS)} curated SNPs found  ({len(findings)/len(COMPREHENSIVE_SNPS)*100:.0f}% coverage)")

In [ ]:
plot_curated_impact(findings)

### 3.2 — High-impact findings (magnitude ≥ 2)

In [ ]:
print_high_impact_findings(findings, min_magnitude=2)

---
## 4 — ClinVar Disease Risk Scan

The curated database covers ~200 well-known variants. ClinVar covers **340,000** clinically annotated variants — including rare ones you'd never think to curate manually.

For each ClinVar row we check if your `chrom:pos` matches, then verify you carry the alternate allele. Indels are skipped — 23andMe chips can't reliably represent multi-base changes.

In [ ]:
from analyze_dna.utils import ensure_clinvar

CLINVAR_PATH = Path(ensure_clinvar(DATA_DIR))
print(f"ClinVar path : {CLINVAR_PATH}")
print(f"Exists       : {CLINVAR_PATH.exists()}")
if CLINVAR_PATH.exists():
    print(f"Size         : {CLINVAR_PATH.stat().st_size / 1_048_576:.0f} MB")

In [ ]:
if CLINVAR_PATH.exists() and genome_by_position:
    print("Scanning ClinVar… (15–25 sec)")
    t0 = time.time()
    clinvar_findings, clinvar_stats = scan_clinvar(CLINVAR_PATH, genome_by_position)
    print(f"Done in {time.time()-t0:.1f}s — {clinvar_stats['total']:,} variants scanned, "
          f"{clinvar_stats['pathogenic']} pathogenic, {clinvar_stats['likely_pathogenic']} likely pathogenic")
else:
    print("ClinVar not available — run: uv run analyze-dna update-clinvar")
    clinvar_findings = {k: [] for k in ["pathogenic", "likely_pathogenic", "risk_factor",
                                         "drug_response", "protective", "other"]}
    clinvar_stats = {}

### 4.1 — Findings by category

In [ ]:
plot_clinvar_categories(clinvar_findings)

### 4.2 — Pathogenic findings detail

Zygosity matters: one copy of a recessive variant makes you a **carrier**; two copies (or one copy of a dominant variant) means **affected**.

In [ ]:
display_pathogenic_findings(clinvar_findings)

---
## 5 — PharmGKB Drug Interactions

PharmGKB maps specific genotypes to drug response predictions.
Unlike ClinVar (disease risk), PharmGKB focuses entirely on **how your genetics
affect medications** — dosing, efficacy, and toxicity.

In [ ]:
from analyze_dna.utils import load_pharmgkb

pharmgkb = load_pharmgkb(
    DATA_DIR / "clinical_annotations.tsv",
    DATA_DIR / "clinical_ann_alleles.tsv",
)
print(f"PharmGKB SNPs loaded: {len(pharmgkb):,}")

In [ ]:
drug_interactions = find_drug_interactions(genome_by_rsid, pharmgkb)
print(f"{len(drug_interactions)} drug interaction annotations found")

### 5.1 — Evidence level distribution

Level 1A = clinical guideline (act on it). Level 4 = preliminary (informational only).

In [ ]:
plot_pharmgkb_levels(drug_interactions)

### 5.2 — Highest-confidence drug interactions (level 1A / 1B)

In [ ]:
display_drug_interactions(drug_interactions, levels=("1A", "1B"))

---
## 6 — Summary Dashboard

In [ ]:
plot_summary_dashboard(genome_by_rsid, findings, clinvar_findings, drug_interactions)

---
## 7 — Ancestry Composition

Using ~35 ancestry-informative markers (AIMs) with known population allele frequencies from **gnomAD v4**, we estimate which continental population your genotypes most closely match.

**Method:** Hardy-Weinberg log-likelihood — for each marker, compute P(your genotype | population) assuming HWE, accumulate log-probabilities across all markers, then normalise via softmax. This is the same basic approach as 23andMe's ancestry inference, just with fewer markers.

**First run** queries the gnomAD API (~30 sec). All subsequent runs use the local cache.

In [ ]:
from analyze_dna.ancestry import (
    download_aim_frequencies, load_aim_frequencies, compute_ancestry, ANCESTRY_AIMS
)

# Download frequencies from gnomAD (first run only — ~30 sec; cached afterwards)
freq_path = DATA_DIR / "ancestry" / "aim_frequencies.json"
if not freq_path.exists():
    print(f"First run: querying gnomAD for {len(ANCESTRY_AIMS)} markers…")
    download_aim_frequencies(DATA_DIR)

frequencies = load_aim_frequencies(DATA_DIR)
print(f"{len(frequencies)} ancestry-informative markers loaded from cache")

In [ ]:
ancestry_probs, n_aim_markers = compute_ancestry(genome_by_rsid, frequencies)
plot_ancestry_composition(ancestry_probs, n_markers=n_aim_markers, subject_name=GENOME_PATH.stem)

---
## Key Takeaways

| Step | What we did | Key insight |
|------|------------|-------------|
| Load genome | Parse TSV → two hash maps | `chrom:pos` key enables O(1) matching against any database |
| Individual lookup | Direct rsID lookup | Single dict lookup — it's really that simple |
| Curated analysis | Match ~200 pre-interpreted SNPs | Most actionable results come from well-studied variants |
| ClinVar scan | Scan ~340K clinical variants | Always filter indels; always show gold stars |
| PharmGKB | Drug interaction annotations | Level 1A findings have clinical guideline backing |
| Full pipeline | CLI wraps everything | Reports in <30 seconds, fully local |

## Resources

| Resource | What it is |
|----------|-----------|
| `ncbi.nlm.nih.gov/clinvar` | ClinVar — clinical variant database |
| `pharmgkb.org` | PharmGKB — drug-gene interactions |
| `cpicpgx.org` | CPIC — clinical PGx guidelines |
| `ncbi.nlm.nih.gov/snp` | dbSNP — SNP reference |
| `gnomad.broadinstitute.org` | gnomAD — population allele frequencies |
| `omim.org` | OMIM — gene-disease associations |
| `pharmvar.org` | PharmVar — CYP star allele nomenclature |
| `acmg.net/guidelines` | ACMG variant classification guidelines |